In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch, json, os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

LABEL_NAMES = {0:"Yardım Talebi",1:"Kayıp Bildirimi",2:"Altyapı Hasarı",
               3:"Bağış/Koordinasyon",4:"Diğer/İlgisiz"}
train_df = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/train.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/test.csv')

# Her sınıftan 1 gösterim örneği (train'den — sızıntı yok)
demos = [(train_df[train_df.label_id==c].iloc[0]['tweet'], LABEL_NAMES[c]) for c in range(5)]
print("Gösterim örnekleri:")
for t, lab in demos: print(f"  [{lab}] {t[:50]}...")

def build_fewshot(tweet):
    s = ("Sen bir deprem acil durum sınıflandırma asistanısın. "
         "Verilen tweeti aşağıdaki kategorilerden birine koy. SADECE kategori adını yaz.\n\n"
         "Kategoriler: Yardım Talebi, Kayıp Bildirimi, Altyapı Hasarı, Bağış/Koordinasyon, Diğer/İlgisiz\n\n"
         "Örnekler:\n")
    for t, lab in demos:
        s += f"Tweet: {t}\nKategori: {lab}\n\n"
    s += f"Şimdi sınıflandır:\nTweet: {tweet}\nKategori:"
    return s

def parse_label(text):
    t = text.lower()
    if "yardım" in t or "yardim" in t:                      return 0
    if "kayıp" in t or "kayip" in t:                        return 1
    if "altyapı" in t or "altyapi" in t:                    return 2
    if "bağış" in t or "bagis" in t or "koordinasyon" in t: return 3
    if "diğer" in t or "diger" in t or "ilgisiz" in t:      return 4
    return -1

MID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
mdl = AutoModelForCausalLM.from_pretrained(MID, device_map="auto", dtype=torch.float16)
mdl.eval()

preds, trues, fails = [], [], 0
for _, row in test_df.iterrows():
    msgs = [{"role":"user","content":build_fewshot(row['tweet'])}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(mdl.device)
    n = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=12, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][n:], skip_special_tokens=True).strip()
    pid = parse_label(gen)
    if pid == -1: fails += 1
    preds.append(pid if pid!=-1 else 4); trues.append(int(row['label_id']))

acc = accuracy_score(trues, preds)
mf1 = f1_score(trues, preds, average='macro', zero_division=0)
wf1 = f1_score(trues, preds, average='weighted', zero_division=0)
print(f"\n{'='*55}\nQwen2.5-1.5B — FEW-SHOT (5 gösterim) SONUÇLARI\n{'='*55}")
print(f"Accuracy    : {acc:.4f} ({acc*100:.1f}%)")
print(f"Macro-F1    : {mf1:.4f}\nWeighted-F1 : {wf1:.4f}")
print(f"Parse fail  : {fails}/{len(test_df)}\n")
print(classification_report(trues, preds, labels=[0,1,2,3,4],
      target_names=[LABEL_NAMES[i] for i in range(5)], zero_division=0))

out_dir = '/content/drive/MyDrive/bdm_proje/results/few_shot'; os.makedirs(out_dir, exist_ok=True)
sonuc = {"model":"Qwen2.5-1.5B-Instruct","yontem":"Few-shot (5 gösterim, her sınıftan 1)",
         "test_size":int(len(test_df)),"accuracy":round(float(acc),4),
         "f1_macro":round(float(mf1),4),"f1_weighted":round(float(wf1),4),
         "parse_basarisiz":int(fails),
         "confusion_matrix":confusion_matrix(trues,preds,labels=[0,1,2,3,4]).tolist()}
with open(os.path.join(out_dir,'qwen25_few_shot_sonuc.json'),'w',encoding='utf-8') as f:
    json.dump(sonuc, f, ensure_ascii=False, indent=2)
print(f"✅ Kaydedildi → {out_dir}/qwen25_few_shot_sonuc.json")

Gösterim örnekleri:
  [Yardım Talebi] Beyoğlu mahallesi, şehit İsmail Orçan bulvarı no:3...
  [Kayıp Bildirimi] Kardeşim market çalışanı, mağaza yıkılmış. İçeride...
  [Altyapı Hasarı] Hatay Atatürk stadyumu çökmüş, toplanma alanı olar...
  [Bağış/Koordinasyon] Depremzedeler için ücretsiz taşıma hizmeti veriyor...
  [Diğer/İlgisiz] Rant uğruna bina yoğını yaptığınız şehre karşı ken...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Qwen2.5-1.5B — FEW-SHOT (5 gösterim) SONUÇLARI
Accuracy    : 0.6000 (60.0%)
Macro-F1    : 0.6256
Weighted-F1 : 0.6060
Parse fail  : 0/80

                    precision    recall  f1-score   support

     Yardım Talebi       0.68      0.66      0.67        29
   Kayıp Bildirimi       0.69      0.90      0.78        10
    Altyapı Hasarı       1.00      1.00      1.00        10
Bağış/Koordinasyon       0.19      0.36      0.25        11
     Diğer/İlgisiz       0.75      0.30      0.43        20

          accuracy                           0.60        80
         macro avg       0.66      0.64      0.63        80
      weighted avg       0.67      0.60      0.61        80

✅ Kaydedildi → /content/drive/MyDrive/bdm_proje/results/few_shot/qwen25_few_shot_sonuc.json
